## This notebook gathers albedo, relight and NVS results across all synthetic scenes and prints mean values.

## Print material

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# List your base paths here
base_paths = [
    "path/to/output_synthetic",    
]

def compute_means(subfolder_path):
    psnr_vals, ssim_vals, lpips_vals, rough_vals = [], [], [], []

    for scene_name in os.listdir(subfolder_path):
        scene_path = os.path.join(subfolder_path, scene_name)

        if "" in scene_path:  #you can also specify singe scene name, eg. "mouse"
            json_file = os.path.join(scene_path, "results_material_static.json")
            if os.path.isfile(json_file):
                with open(json_file, "r") as f:
                    data = json.load(f)
                    psnr_vals.append(data.get("psnr_albedo_avg", 0))
                    ssim_vals.append(data.get("ssim_albedo_avg", 0))
                    lpips_vals.append(data.get("lpips_albedo_avg", 0))
                    rough_vals.append(data.get("mse_roughness_avg", 0))

    if psnr_vals:
        return {
            "psnr": np.mean(psnr_vals),
            "ssim": np.mean(ssim_vals),
            "lpips": np.mean(lpips_vals),
            "mse_rough": np.mean(rough_vals),
            "psnr_std": np.std(psnr_vals),
            "ssim_std": np.std(ssim_vals),
            "lpips_std": np.std(lpips_vals),
            "mse_rough_std": np.std(rough_vals)
        }
    return None

# Collect data into nested dict: {subfolder: {metric: {base_name: value}}}
data = {}

for base_path in base_paths:
    base_name = os.path.basename(base_path)
    if not os.path.isdir(base_path):
        print(f"Skipping missing path: {base_path}")
        continue

    for subfolder in os.listdir(base_path):
        subfolder_path = os.path.join(base_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue

        metrics = compute_means(subfolder_path)
        if metrics:
            if subfolder not in data:
                data[subfolder] = {"psnr": {}, "ssim": {}, "lpips": {}, "mse_rough": {}, 
                                   "psnr_std": {}, "ssim_std": {}, "lpips_std": {}, "mse_rough_std": {}}
            for metric_name, value in metrics.items():
                data[subfolder][metric_name][base_name] = value

# Convert to DataFrame with multi-row index
rows = []
index = []
for subfolder, metrics in data.items():
    for metric in ["psnr", "ssim", "lpips", "mse_rough", "psnr_std", "ssim_std", "lpips_std", "mse_rough_std"]:
        row = []
        for base_path in base_paths:
            base_name = os.path.basename(base_path)
            row.append(metrics[metric].get(base_name, np.nan))
        rows.append(row)
        index.append((subfolder, metric))

multi_index = pd.MultiIndex.from_tuples(index, names=["scene", "metric"])
df = pd.DataFrame(rows, index=multi_index, columns=[os.path.basename(p) for p in base_paths])
df


outputs_specular32_newenv_best_inter_optim_v6_with_envloss
scene               metric                                                                   
goldenbay_damwall   psnr                                                   27.928623         
                    ssim                                                    0.959408         
                    lpips                                                   0.058234         
                    mse_rough                                               0.009885         
                    psnr_std                                                1.931633         
                    ssim_std                                                0.010017         
                    lpips_std                                               0.018840         
                    mse_rough_std                                           0.002229         
damwall_harbour     psnr                                                   27.268440         
                    ssim                                                    0.952042         
                    lpips                                                   0.069295         
                    mse_rough                                               0.011533         
                    psnr_std                                                1.568480         
                    ssim_std                                                0.007420         
                    lpips_std                                               0.023710         
                    mse_rough_std                                           0.002333         
chapelday_goldenbay psnr                                                   30.838716         
                    ssim                                                    0.973387         
                    lpips                                                   0.035933         
                    mse_rough                                               0.011025         
                    psnr_std                                                1.797643         
                    ssim_std                                                0.007255         
                    lpips_std                                               0.014456         
                    mse_rough_std                                           0.002161

## Print relight

In [ ]:
import os
import json
import numpy as np
import pandas as pd


base_paths = [
    "path/to/output_synthetic"
]

# Scene name prefixes to include
def compute_means(folder):
    psnr_vals, ssim_vals, lpips_vals = [], [], []

    for scene_name in os.listdir(folder):

        scene_path = os.path.join(folder, scene_name)

        if "" in scene_path: #you can also specify singe scene name, eg. "mouse"
            json_file = os.path.join(scene_path, "results_relight_static.json")
            if os.path.isfile(json_file):
                with open(json_file, "r") as f:
                    data = json.load(f)
                    psnr_vals.append(data.get("psnr_pbr_avg", 0))
                    ssim_vals.append(data.get("ssim_pbr_avg", 0))
                    lpips_vals.append(data.get("lpips_pbr_avg", 0))

    if psnr_vals:
        return {
            "psnr": np.mean(psnr_vals),
            "ssim": np.mean(ssim_vals),
            "lpips": np.mean(lpips_vals),
            "psnr_std": np.std(psnr_vals),
            "ssim_std": np.std(ssim_vals),
            "lpips_std": np.std(lpips_vals)
        }
    return None

# {scene: {metric: {base_name: value}}}
data = {}

for base_path in base_paths:
    base_name = os.path.basename(base_path)
    if not os.path.isdir(base_path):
        print(f"Skipping missing base path: {base_path}")
        continue

    # Search nested subfolders recursively
    for root, dirs, files in os.walk(base_path):
        for subfolder in dirs:

            subfolder_path = os.path.join(root, subfolder)
            if not os.path.isdir(subfolder_path):
                continue

            metrics = compute_means(subfolder_path)
            if metrics:
                if subfolder not in data:
                    data[subfolder] = {"psnr": {}, "ssim": {}, "lpips": {}, "psnr_std": {}, "ssim_std": {}, "lpips_std": {}}
                for metric_name, value in metrics.items():
                    data[subfolder][metric_name][base_name] = value

# Build DataFrame
rows = []
index = []
for subfolder, metrics in data.items():
    for metric in ["psnr", "ssim", "lpips", "psnr_std", "ssim_std", "lpips_std"]:
        row = []
        for base_path in base_paths:
            base_name = os.path.basename(base_path)
            row.append(metrics[metric].get(base_name, np.nan))
        rows.append(row)
        index.append((subfolder, metric))

multi_index = pd.MultiIndex.from_tuples(index, names=["scene", "metric"])
df = pd.DataFrame(rows, index=multi_index, columns=[os.path.basename(p) for p in base_paths])
df.round(3)


outputs_specular32_newenv_best_inter_optim_v6_with_envloss
scene               metric                                                               
goldenbay_damwall   psnr                                                  25.854         
                    ssim                                                   0.938         
                    lpips                                                  0.045         
                    psnr_std                                               0.779         
                    ssim_std                                               0.011         
                    lpips_std                                              0.009         
damwall_harbour     psnr                                                  26.037         
                    ssim                                                   0.928         
                    lpips                                                  0.060         
                    psnr_std                                               0.579         
                    ssim_std                                               0.007         
                    lpips_std                                              0.012         
chapelday_goldenbay psnr                                                  28.563         
                    ssim                                                   0.939         
                    lpips                                                  0.041         
                    psnr_std                                               0.478         
                    ssim_std                                               0.011         
                    lpips_std                                              0.007

## Print nvs

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# List your base paths here
base_paths = [
    "path/to/output_synthetic"
]

def compute_means(subfolder_path):
    psnr_vals, ssim_vals, lpips_vals = [], [], []

    for scene_name in os.listdir(subfolder_path):
        scene_path = os.path.join(subfolder_path, scene_name)
        if "" in scene_path:  #you can also specify singe scene name, eg. "mouse"
            json_file = os.path.join(scene_path, "results_nvs_static.json")
            if os.path.isfile(json_file):
                with open(json_file, "r") as f:
                    data = json.load(f)
                    psnr_vals.append(data.get("psnr_nvs_avg", 0))
                    ssim_vals.append(data.get("ssim_nvs_avg", 0))
                    lpips_vals.append(data.get("lpips_nvs_avg", 0))

    if psnr_vals:
        return {
            "psnr": np.mean(psnr_vals),
            "ssim": np.mean(ssim_vals),
            "lpips": np.mean(lpips_vals)
        }
    return None

# Collect data into nested dict: {subfolder: {metric: {base_name: value}}}
data = {}

for base_path in base_paths:
    base_name = os.path.basename(base_path)
    if not os.path.isdir(base_path):
        print(f"Skipping missing path: {base_path}")
        continue

    for subfolder in os.listdir(base_path):
        subfolder_path = os.path.join(base_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue

        metrics = compute_means(subfolder_path)
        if metrics:
            if subfolder not in data:
                data[subfolder] = {"psnr": {}, "ssim": {}, "lpips": {}}
            for metric_name, value in metrics.items():
                data[subfolder][metric_name][base_name] = value

# Convert to DataFrame with multi-row index
rows = []
index = []
for subfolder, metrics in data.items():
    for metric in ["psnr", "ssim", "lpips"]:
        row = []
        for base_path in base_paths:
            base_name = os.path.basename(base_path)
            row.append(metrics[metric].get(base_name, np.nan))
        rows.append(row)
        index.append((subfolder, metric))

multi_index = pd.MultiIndex.from_tuples(index, names=["light condition", "metric"])
df = pd.DataFrame(rows, index=multi_index, columns=[os.path.basename(p) for p in base_paths])
df.round(3)


outputs_specular32_newenv_best_inter_optim_v6_with_envloss
light condition     metric                                                            
goldenbay_damwall   psnr                                               29.859         
                    ssim                                                0.954         
                    lpips                                               0.023         
damwall_harbour     psnr                                               26.948         
                    ssim                                                0.952         
                    lpips                                               0.025         
chapelday_goldenbay psnr                                               27.636         
                    ssim                                                0.952         
                    lpips                                               0.022